In [20]:
import torch
import pandas as pd
import numpy as np
from torch import optim,nn
from torch.utils.data import DataLoader,Dataset,TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_iris

In [4]:
X,y = load_iris(return_X_y=True)
X.shape,y.shape

((150, 4), (150,))

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X,y,random_state=12,test_size=0.2,stratify=y)
X_train.shape , X_test.shape , y_train.shape , y_test.shape

((120, 4), (30, 4), (120,), (30,))

In [14]:
X_train , X_valid , y_train , y_valid = train_test_split(X_train,y_train,random_state=12,test_size=0.1,stratify=y_train)

In [15]:
X_scaler = StandardScaler()
X_train = X_scaler.fit_transform(X_train)
X_test = X_scaler.transform(X_test)
X_valid = X_scaler.transform(X_valid)

In [16]:
num_samples, num_features = X.shape
num_samples,num_features

(150, 4)

In [18]:
num_classes = len(np.unique(y))
num_classes

3

In [19]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_valid = torch.tensor(X_valid, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.int64)
y_valid = torch.tensor(y_valid, dtype=torch.int64)
y_test = torch.tensor(y_test, dtype=torch.int64)

In [21]:
train_set = TensorDataset(X_train, y_train)
valid_set = TensorDataset(X_valid, y_valid)
test_set = TensorDataset(X_test, y_test)

In [23]:
train_loader = DataLoader(train_set,batch_size=10, shuffle=True)
valid_loader = DataLoader(valid_set,batch_size=5, shuffle=True)
test_loader = DataLoader(test_set,batch_size=200, shuffle=True)

## Model

In [59]:
model = nn.Linear(num_features,num_classes)
model

Linear(in_features=4, out_features=3, bias=True)

In [60]:
loss_fn = nn.CrossEntropyLoss()

In [61]:
optimizer = optim.SGD(model.parameters(),lr= 0.1,momentum=0.9)

In [62]:
X_batch, y_batch = next(iter(train_loader))
y_hat = model(X_batch)
y_hat.shape , y_batch.shape , X_batch.shape

(torch.Size([10, 3]), torch.Size([10]), torch.Size([10, 4]))

In [63]:
loss = loss_fn(y_hat,y_batch)
loss

tensor(1.2245, grad_fn=<NllLossBackward0>)

In [65]:
epoches_num = 100
train_hist_loss = []
train_hist_acc = []
valid_hist_loss = []
valid_hist_acc = []
best_model_acc = torch.inf

for epoch in range(epoches_num):
    mean_loss_train , mean_loss_valid = 0 , 0
    mean_acc_train , mean_acc_valid = 0 , 0

    for X_batch, y_batch in train_loader:
        y_hat = model(X_batch)
        loss = loss_fn(y_hat , y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        mean_loss_train += loss.item() * len(X_batch)
        mean_acc_train += torch.sum(y_hat.argmax(dim=1) == y_batch).item()

    mean_loss_train = mean_loss_train / len(train_set)
    mean_acc_train = mean_acc_train / len(train_set)

    train_hist_loss.append(mean_loss_train)
    train_hist_acc.append(mean_loss_train)

    print(f'Epoch {epoch+1}/{epoches_num} | Train Loss: {mean_loss_train:.4f} | Train Accuracy: {mean_acc_train:.4f}')

    with torch.no_grad():
        for X_batch, y_batch in valid_loader:
            y_hat = model(X_batch)
            loss = loss_fn(y_hat,y_batch)
            mean_loss_valid += loss.item() * len(X_batch)
            mean_acc_valid += torch.sum(y_hat.argmax(dim=1) == y_batch).item()

    mean_loss_valid = mean_loss_valid / len(valid_set)
    mean_acc_valid = mean_acc_valid / len(valid_set)

    valid_hist_loss.append(mean_loss_valid)
    valid_hist_acc.append(mean_loss_valid)

    if mean_loss_valid < best_model_acc:
        best_model_acc = mean_loss_valid
        torch.save(model, 'beat_model.pt')

    print(f'Epoch {epoch+1}/{epoches_num} | Validation Loss: {mean_loss_valid:.4f} | Validation Accuracy: {mean_acc_valid:.4f}')

Epoch 1/100 | Train Loss: 0.0579 | Train Accuracy: 0.9722
Epoch 1/100 | Validation Loss: 0.1206 | Validation Accuracy: 1.0000
Epoch 2/100 | Train Loss: 0.0585 | Train Accuracy: 0.9722
Epoch 2/100 | Validation Loss: 0.1131 | Validation Accuracy: 1.0000
Epoch 3/100 | Train Loss: 0.0594 | Train Accuracy: 0.9722
Epoch 3/100 | Validation Loss: 0.1277 | Validation Accuracy: 1.0000
Epoch 4/100 | Train Loss: 0.0586 | Train Accuracy: 0.9722
Epoch 4/100 | Validation Loss: 0.1029 | Validation Accuracy: 1.0000
Epoch 5/100 | Train Loss: 0.0577 | Train Accuracy: 0.9722
Epoch 5/100 | Validation Loss: 0.1110 | Validation Accuracy: 1.0000
Epoch 6/100 | Train Loss: 0.0599 | Train Accuracy: 0.9722
Epoch 6/100 | Validation Loss: 0.0970 | Validation Accuracy: 1.0000
Epoch 7/100 | Train Loss: 0.0591 | Train Accuracy: 0.9722
Epoch 7/100 | Validation Loss: 0.1190 | Validation Accuracy: 1.0000
Epoch 8/100 | Train Loss: 0.0583 | Train Accuracy: 0.9722
Epoch 8/100 | Validation Loss: 0.1053 | Validation Accuracy:

In [72]:
model = torch.load('./beat_model.pt', weights_only=False)
with torch.no_grad():
    for X_batch , y_batch in test_loader:
        y_hat = model(X_batch)
        loss = torch.sum(loss_fn(y_hat, y_batch)) / len(X_batch)
        acc = torch.sum(y_hat.argmax(dim=1) == y_batch).item() / len(X_batch)

print(f'Test Loss: {loss:.4f} | Test Accuracy: {acc:.4f}')

Test Loss: 0.0006 | Test Accuracy: 1.0000
